In [1]:
import datetime
import io
import sys
import os
from pathlib import Path
import unicodedata
from datetime import timezone

import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import pybaseball
import requests
import scipy.stats as stats
from bs4 import BeautifulSoup
from pybaseball import statcast
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
from sklearn.calibration import calibration_curve

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src' / 'mlb_props').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Run this notebook from somewhere inside the MLB-Props repository.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from mlb_props.config import PROCESSED_DATA_DIR, ROSTER_SCRAPER_DIR, SAVANT_DATA_DIR

sys.path.insert(0, str(ROSTER_SCRAPER_DIR))
from RosterScraper import RosterScraper

In [2]:
# StatCast player ID map
url = 'https://docs.google.com/spreadsheets/d/1JgczhD5VDQ1EiXqVG-blttZcVwbZd5_Ne_mefUGwJnk/pub?output=csv'
res = requests.get(url)
ID = pd.read_csv(io.BytesIO(res.content), sep=',')
ID.dropna(subset=['MLBID'], inplace=True)
ID['MLBID'] = ID['MLBID'].astype(int)

# 40-man roster scrape
Rosters = RosterScraper()
BID = Rosters[Rosters['Position'] == 'Batter']
PID = Rosters[Rosters['Position'] == 'Pitcher']

# --- Helper Functions ---
def convert_name(name):
    name_map = {
        'Rockies': 'COL', 'Reds': 'CIN', 'Mariners': 'SEA', 'Nationals': 'WSH',
        'Yankees': 'NYY', 'Astros': 'HOU', 'Red Sox': 'BOS', 'Athletics': 'OAK',
        'Mets': 'NYM', 'Braves': 'ATL', 'Giants': 'SF', 'Brewers': 'MIL',
        'Rays': 'TB', 'Royals': 'KC', 'White Sox': 'CWS', 'Cubs': 'CHC',
        'Angels': 'LAA', 'Tigers': 'DET', 'Diamondbacks': 'ARI', 'Guardians': 'CLE',
        'Orioles': 'BAL', 'Twins': 'MIN', 'Marlins': 'MIA', 'Phillies': 'PHI',
        'Rangers': 'TEX', 'Dodgers': 'LAD', 'Padres': 'SD', 'Pirates': 'PIT',
        'Blue Jays': 'TOR', 'Cardinals': 'STL'
    }
    return name_map.get(name, np.nan)

def flip_names(name):
    if ',' in name:
        last, first = name.split(', ', 1)
        return f"{first} {last}"
    return name

def replace_special_chars(text):
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')

def append_suffix_to_duplicates(df, column):
    seen = {}
    for idx, value in enumerate(df[column]):
        if value in seen:
            seen[value] += 1
            df.at[idx, column] = f"{value}2"
        else:
            seen[value] = 1

In [3]:
def getDKData2024():
    eastern_time = datetime.datetime.now(timezone.utc).astimezone(timezone(datetime.timedelta(hours=-5)))
    todaysdate = eastern_time.strftime("%m-%d-%Y")
    url = 'https://rotogrinders.com/lineups/mlb?site=draftkings'
    r = requests.get(url)
    soup = BeautifulSoup(r.text, 'lxml')

    gamelist = []
    gamecards = soup.findAll("div", {"class": "game-card-teams"})
    for x in gamecards:
        twoteams = x.findAll("span", {"class": "team-nameplate-mascot"})
        roadteam = convert_name(twoteams[0].text)
        hometeam = convert_name(twoteams[1].text)
        gamekey = "{}@{}".format(roadteam,hometeam)
        gamelist.append(gamekey)

    matchupsdf = pd.DataFrame()
    for game in gamelist:
        roadteam = game.split("@")[0]
        hometeam = game.split("@")[1]
        thisdf1 = pd.DataFrame({"Team": roadteam, "Opp": hometeam, "RoadTeam": roadteam, "HomeTeam": hometeam},index=[0])
        thisdf2 = pd.DataFrame({"Team": hometeam, "Opp": roadteam, "RoadTeam": roadteam, "HomeTeam": hometeam},index=[0])
        matchupsdf = pd.concat([matchupsdf,thisdf1,thisdf2])
        
    oppdict = dict(zip(matchupsdf.Team,matchupsdf.Opp))
    hometeamdict = dict(zip(matchupsdf.Team,matchupsdf.HomeTeam))
    roadteamdict = dict(zip(matchupsdf.Team,matchupsdf.RoadTeam))

    disabled_span_list = []
    for span in soup.findAll("span", {"class": "player-nameplate disabled"}):
        for a in span.findAll("a"):
            disabled_span_list.append(a.text)

    spdata = pd.DataFrame()
    for div in soup.findAll("span", {"class": "player-nameplate", "data-position": "SP"}):
        if "TBD" in str(div):
            playername = "TBD"
            pos = "SP"
            sal = 0
        else:
            for a in div.findAll('a', {'class': 'player-nameplate-name'}):
                playername = a.text.strip()

            strdiv = str(div)
            pos = strdiv[strdiv.find("data-position")+15:strdiv.find("data-salary")-2]
            sal = strdiv[strdiv.find("data-salary")+13:strdiv.find("<div class = 'player-nameplate-info'>")-3]
        try:
            ownership = strdiv[strdiv.find('<span class="small muted" data-auth="502">') + 42:strdiv.find('%')]
            ownership = ownership.replace("</span>", "")
            ownership = ownership.replace("</span", "")
            ownership = ownership.replace("</div>", "")
            ownership = ownership.replace(" ", "")
        except:
            ownership = np.nan

        thisspdata = pd.DataFrame([[playername, sal, ownership]], columns = ["Player", "Salary", "Ownership"])
        spdata = pd.concat([spdata, thisspdata])

    spdata['Player'] = spdata['Player'].replace('Luis Ortiz', 'Luis L. Ortiz')
    spdata['Player'] = spdata['Player'].replace('Mike King', 'Michael King')
    spdata['Player'] = spdata['Player'].replace('Robert Zastryzny', 'Rob Zastryzny')

    spdata2 = pd.merge(spdata, PID[["Name", "Team"]], left_on = ["Player"], right_on = ["Name"], how = "left").rename(columns = {"Team": "PitcherTeam"})
    spdata3 = pd.merge(spdata2, matchupsdf[["Team", "Opp"]], left_on = ["PitcherTeam"], right_on = ["Team"], how = "left").drop(columns = ["Team"])

    append_suffix_to_duplicates(spdata3, 'PitcherTeam')
    append_suffix_to_duplicates(spdata3, 'Opp')

    opp_spname_dict = dict(zip(spdata3.Opp, spdata3.Player))
    opp_spsal_dict = dict(zip(spdata3.Opp, spdata.Salary))
    opp_spown_dict = dict(zip(spdata3.Opp, spdata3.Ownership))

    ludf = pd.DataFrame()
    
    for li in soup.findAll("li", {"class": "lineup-card-player"}):
        for a in li.findAll("a", {"class": ["player-nameplate-name", "player-nameplate disabled"]}):
            playername = a.text

        listring = str(li)
        for span in li.find("span", {"class": "small"}):
            luspot = span.text
            luspot = luspot.replace("\n", "")
            luspot = luspot.strip()
            luspot = int(luspot)
        pos = listring[listring.find("data-position")+15:listring.find("data-salary")-2]
        sal = listring[listring.find("data-salary")+13:listring.find("<span class='small'>")-3]
        ownership = ownership.replace("</span>", "")
        ownership = ownership.replace("</span", "")
        ownership = ownership.replace("</li", "")
        ownership = ownership.replace("</div>", "")
        ownership = ownership.replace(" ", "")

        try:
            sal = int(sal)
        except:
            sal = 0
        thisludf = pd.DataFrame([[playername, luspot, sal, ownership]], columns = ["Player", "Spot", "Sal", "Ownership"])
        ludf = pd.concat([ludf, thisludf])

    ludf2 = pd.merge(ludf, BID[["Name", "Team"]], left_on = ["Player"], right_on = ["Name"], how = "left").rename(columns = {"Team": "BatterTeam"})
    ludf2['BatterTeam'] = ludf2['BatterTeam'].fillna(method='ffill')
    ludf2['HomeTeam'] = ludf2['BatterTeam'].map(hometeamdict)
    ludf2['RoadTeam'] = ludf2['BatterTeam'].map(roadteamdict)

    ludf2_teamlist = list(ludf2["BatterTeam"])

    dhteams = []
    for x in ludf2_teamlist:
        if ludf2_teamlist.count(x) > 11:
            if x in dhteams:
                pass
            else:
                dhteams.append(x)

    extract_dh = ludf2[ludf2["BatterTeam"].isin(dhteams)]
    new_ludf2 = ludf2[~ludf2["BatterTeam"].isin(dhteams)]

    new_team_list = []
    new_home_list = []
    new_road_list = []
    runcounter = 0

    for x, home, road in zip(extract_dh["BatterTeam"].astype(str), 
                         extract_dh["HomeTeam"].astype(str), 
                         extract_dh["RoadTeam"].astype(str)):
        if runcounter < 18:
            new_team_list.append(x)
            new_home_list.append(home)
            new_road_list.append(road)
            runcounter += 1
        else:
            new_team_list.append(x + "2")
            new_home_list.append(home + "2")
            new_road_list.append(road + "2")
            runcounter += 1

    extract_dh["BatterTeam"] = new_team_list
    extract_dh["HomeTeam"] = new_home_list
    extract_dh["RoadTeam"] = new_road_list

    ludf2 = pd.concat([extract_dh, new_ludf2])
    ludf2["Opp"] = ludf2["BatterTeam"].map(oppdict)
    ludf2['SP'] = ludf2['BatterTeam'].map(opp_spname_dict)
    ludf2['SPSal'] = ludf2['BatterTeam'].map(opp_spsal_dict)
    ludf2['SPOwnership'] = ludf2['BatterTeam'].map(opp_spown_dict)
    ludf2['Date'] = todaysdate
    ludf2['Time'] = np.nan

    ludf3 = ludf2[['BatterTeam','RoadTeam','HomeTeam','Time','Spot','Player','Sal','Ownership','Date', "SP"]]

    dkdata = ludf3.copy()

    try:
        checknan = dkdata[["BatterTeam", "SP"]]
        getnans = checknan[["SP"].isna()]
        if len(getnans) == 0:
            nonans = 1
            nanmapdict = {}
        else:
            nonans = 0
            getnans["SP"] = disabled_span_list
            nanmapdict = dict(zip(getnans.Team, getnans.SP))
    except:
        pass

    try:
        dkdata["SP"] = np.where(dkdata["SP"].isna(), dkdata["BatterTeam"].map(nanmapdict), dkdata["SP"])
    except:
        pass
    
    for i in range(1, len(dkdata) - 1):
        if dkdata.loc[i, 'BatterTeam'] != dkdata.loc[i-1, 'BatterTeam']:
            if dkdata.loc[i, 'BatterTeam'] != dkdata.loc[i+1, 'BatterTeam']:
                dkdata.loc[i, 'BatterTeam'] = np.nan
                dkdata.loc[i, 'HomeTeam'] = np.nan
                dkdata.loc[i, 'RoadTeam'] = np.nan
                dkdata.loc[i, 'SP'] = np.nan

    
    dkdata[["BatterTeam", "RoadTeam", "HomeTeam"]] = dkdata[["BatterTeam", "RoadTeam", "HomeTeam"]].fillna(method='ffill')
    dkdata = dkdata.drop_duplicates(subset = ["BatterTeam", "SP"], keep = "first")
    dkdata = dkdata.drop(columns = ["Time", "Sal", "Ownership"])

    dkdata['BatterTeam'] = dkdata['BatterTeam'].replace('ARI', 'AZ')
    dkdata['RoadTeam'] = dkdata['RoadTeam'].replace('ARI', 'AZ')
    dkdata['HomeTeam'] = dkdata['HomeTeam'].replace('ARI', 'AZ')

    dkdata['Date'] = pd.to_datetime(dkdata['Date'])
    dkdata['Date'] = dkdata['Date'].dt.strftime('%Y-%m-%d')
    dkdata = dkdata.set_index("Date")
    dkdata = dkdata[["BatterTeam", "RoadTeam", "HomeTeam", "SP"]]
    dkdata = pl.from_pandas(dkdata)

    return(dkdata)

In [ ]:
base_path = SAVANT_DATA_DIR
years = [2022, 2023, 2024, 2025]

dfs = {}
for year in years:
    year_path = base_path / str(year)
    parquet_files = list(year_path.glob("*.parquet"))
    if parquet_files:
        df = pl.read_parquet(parquet_files[0])
        dfs[year] = df

In [17]:
# ── Step 1: Fix schema mismatches BEFORE concatenation ──────────────────────
schema_fixes = {
    "sv_id":                                    pl.String,
    "bat_speed":                                pl.Float64,
    "swing_length":                             pl.Float64,
    "arm_angle":                                pl.Float64,
    "attack_angle":                             pl.Float64,
    "attack_direction":                         pl.Float64,
    "swing_path_tilt":                          pl.Float64,
    "intercept_ball_minus_batter_pos_x_inches": pl.Float64,
    "intercept_ball_minus_batter_pos_y_inches": pl.Float64,
    "game_date":                                pl.Datetime("us"),
}

for year in dfs.keys():
    dfs[year] = dfs[year].with_columns([
        pl.col(col).cast(dtype, strict=False)
        for col, dtype in schema_fixes.items()
        if col in dfs[year].columns])

# ── Step 2: Concatenate all years ────────────────────────────────────────────
combined = pl.concat(list(dfs.values()), how="diagonal")
print(f"Successfully concatenated! All years shape: {combined.shape}")

# ── Step 3: Add derived columns (requires home/away cols from raw data) ───────
combined = combined.with_columns([
    pl.when(pl.col("inning_topbot") == "Top")
      .then(pl.col("away_team"))
      .otherwise(pl.col("home_team"))
      .alias("BatterTeam"),
    pl.when(pl.col("inning_topbot") == "Top")
      .then(pl.col("home_team"))
      .otherwise(pl.col("away_team"))
      .alias("PitcherTeam"),
    (pl.col("post_away_score") - pl.col("away_score")).alias("AwayRunsScored"),
    (pl.col("post_home_score") - pl.col("home_score")).alias("HomeRunsScored"),])

# ── Step 4: Clean player names (flip_names + replace_special_chars) ──────────
combined = combined.with_columns(
    pl.col("player_name")
      .map_elements(flip_names, return_dtype=pl.String)
      .alias("player_name"))

combined = combined.with_columns(
    pl.col("player_name")
      .map_elements(replace_special_chars, return_dtype=pl.String)
      .alias("player_name"))

# ── Step 5: Sort — MUST come before groupby so .first() picks true starter ───
groupby_cols = ["game_date", "BatterTeam", "away_team", "home_team"]

# ── Step 6: Sort — Top=0, Bot=1 ensures correct alternation ─────────────────
combined_sorted = combined.sort([
    "game_date", "home_team", "away_team",
    "inning", pl.col("inning_topbot").replace({"Top": "0", "Bot": "1"}),
    "at_bat_number", "pitch_number"])

# ── Step 7: Identify the true starter (first pitcher per team per game) ───────
starter_names = (
    combined_sorted
    .group_by(groupby_cols, maintain_order=True)
    .agg(pl.col("player_name").first().alias("starter_name"))
)

batters_per_starter = (
    combined_sorted
    .join(starter_names, on=groupby_cols, how="left")
    .filter(pl.col("player_name") == pl.col("starter_name"))
    .group_by(groupby_cols)
    .agg(pl.col("at_bat_number").n_unique().alias("starter_batters_faced"))
)

starter_names = starter_names.join(batters_per_starter, on=groupby_cols, how="left")
starter_names = starter_names.with_columns(
    (pl.col("starter_batters_faced") < 9).cast(pl.Int8).alias("is_opener_game")
)

# ── Step 8: Join starters back and filter to starter pitches only ─────────────
df_starters_only = (
    combined_sorted
    .join(starter_names, on=groupby_cols, how="left")
    .filter(pl.col("player_name") == pl.col("starter_name"))
    .drop("starter_name")
    .sort(["game_date", "home_team", "away_team",
           "inning", "at_bat_number", "pitch_number"]))

print(f"Final starter-only 1st Inning shape: {df_starters_only.shape}")

Successfully concatenated! All years shape: (2138463, 119)
Final starter-only 1st Inning shape: (1226875, 125)


In [29]:
# ── Step A: Remap rare/offspeed variants → CH ─────────────────────────────────
df_starters_only = df_starters_only.with_columns(
    pl.col('pitch_type').replace({'FO': 'CH', 'FS': 'CH'}).alias('pitch_type'))

# ── Step B: Compute VAA at pitch level ───────────────────────────────────────
df_starters_only = df_starters_only.with_columns([
    ((-pl.col('vy0') - (pl.col('vy0').pow(2) - 2 * pl.col('ay') * (pl.col('release_pos_y') - 1.417)).sqrt())
     / pl.col('ay')).alias('_t_plate')
]).with_columns([
    (pl.col('vz0') + pl.col('az') * pl.col('_t_plate')).alias('_vz_plate'),
    (pl.col('vy0') + pl.col('ay') * pl.col('_t_plate')).alias('_vy_plate'),
]).with_columns([
    ((pl.col('_vz_plate') / pl.col('_vy_plate').abs()).arctan()
     * (180.0 / 3.141592653589793)).alias('vaa')
]).drop(['_t_plate', '_vz_plate', '_vy_plate'])


df_starters_only = df_starters_only.with_columns([
    (pl.col('pfx_z') * 12).alias('pfx_z'),
    (pl.col('pfx_x') * 12).alias('pfx_x'),])

# ── Pitcher Game-Level Aggregation ────────────────────────────────────────────

PITCH_TYPES_FULL = {
    'FF': 'ff',   # Four-Seam Fastball
    'SI': 'si',   # Sinker
    'FC': 'fc',   # Cutter
    'SL': 'sl',   # Slider
    'ST': 'st',   # Sweeper
    'CU': 'cu',   # Curveball
    'CH': 'ch',   # Changeup
}

groupby_game = ['game_date', 'player_name', 'away_team', 'home_team', 'p_throws']

df_game_aggs = (
    df_starters_only
    .sort(['player_name', 'game_date', 'at_bat_number', 'pitch_number'])
    .group_by(groupby_game, maintain_order=True)
    .agg([
        pl.col("is_opener_game").first().alias("is_opener_game"),
        pl.col("game_date").first().dt.year().cast(pl.Int32).alias("season"),
        (pl.col("game_date").first().dt.year() >= 2026).cast(pl.Int8).alias("abs_era"),

        # ── Totals ────────────────────────────────────────────────────────────
        (
        pl.col('events').is_in([
            'field_out', 'strikeout', 'force_out', 'sac_fly', 'sac_bunt',
            'fielders_choice_out', 'caught_stealing_2b', 'caught_stealing_3b',
            'caught_stealing_home', 'pickoff_1b', 'pickoff_2b', 'pickoff_3b',
            'other_out',
        ]).sum() +
        pl.col('events').is_in(['grounded_into_double_play', 'double_play', 'strikeout_double_play']).sum() * 2 +
        pl.col('events').is_in(['triple_play']).sum() * 3)
            .alias('Outs'),
        pl.col('description').count()
            .alias('Pitches'),
        pl.col('at_bat_number').n_unique()
            .alias('PA'),
        pl.when(pl.col("inning_topbot").first() == "Top")
            .then(pl.col("post_home_score").last() - pl.col("home_score").first())  # away pitcher → home team scored
            .otherwise(pl.col("post_away_score").last() - pl.col("away_score").first())
            .alias("Runs"),
        pl.col('events').is_in(['single', 'double', 'triple', 'home_run']).sum()
            .alias('Hits'),
        pl.col('events').is_in(['strikeout', 'strikeout_double_play']).sum()
            .alias('K'),
        pl.col('events').is_in(['walk']).sum()
            .alias('BB'),
        pl.col('events').is_in(['hit_by_pitch']).sum()
            .alias('HBP'),
        pl.col('events').is_in(['home_run']).sum()
            .alias('HR'),
        pl.col('bb_type').is_in(['fly_ball', "pop_up"]).sum()
            .alias('FB'),
        pl.col('bb_type').is_in(['ground_ball']).sum()
            .alias('GB'),
        pl.col('type').is_in(['S']).sum()
            .alias('Strikes'),
        pl.col('type').is_in(['B']).sum()
            .alias('Balls'),
        pl.col('type').is_in(['X']).sum()
            .alias('BIP'),
        pl.col('description').is_in(['swinging_strike', 'swinging_strike_blocked']).sum()
            .alias('Whiffs'),
        pl.col('description').is_in(['called_strike']).sum()
            .alias('CS'),
        (pl.col('description').is_in(['called_strike', 'swinging_strike', 'swinging_strike_blocked'])).sum()
            .alias('CSW'),

        # ── Averages ─────────────────────────────────────────────────────────
        pl.col('estimated_ba_using_speedangle').mean().alias('xBA'),
        pl.col('woba_value').mean().alias('wOBA'),
        pl.col('estimated_woba_using_speedangle').mean().alias('xwOBA'),

                # ── Binary pitch type flags ───────────────────────────────────────────
        *[
            (pl.col('pitch_type').filter(pl.col('pitch_type') == code).count() > 0)
            .cast(pl.Int8)
            .alias(f'throws_{abbr}')
            for code, abbr in PITCH_TYPES_FULL.items()
        ],

        # ── Per pitch type: velo, IVB (pfx_z), HB (pfx_x), VAA ──────────────
        *[
            expr
            for code, abbr in PITCH_TYPES_FULL.items()
            for expr in [
                pl.col('release_speed').filter(pl.col('pitch_type') == code).mean().alias(f'{abbr}_velo'),
                pl.col('release_spin_rate').filter(pl.col('pitch_type') == code).mean().alias(f'{abbr}_spinrate'),
                pl.col('pfx_z').filter(pl.col('pitch_type') == code).mean().alias(f'{abbr}_ivb'),
                pl.col('pfx_x').filter(pl.col('pitch_type') == code).mean().alias(f'{abbr}_hb'),
                pl.col('vaa').filter(pl.col('pitch_type') == code).mean().alias(f'{abbr}_vaa'),
            ]
        ],

        # ── Usage vs RHB ──────────────────────────────────────────────────────
        *[
            (pl.col('pitch_type').filter(
                (pl.col('pitch_type') == code) & (pl.col('stand') == 'R')
             ).count() /
             pl.col('pitch_type').filter(pl.col('stand') == 'R').count().clip(lower_bound=1)
            ).alias(f'{abbr}_usage_vR')
            for code, abbr in PITCH_TYPES_FULL.items()
        ],

        # ── Usage vs LHB ──────────────────────────────────────────────────────
        *[
            (pl.col('pitch_type').filter(
                (pl.col('pitch_type') == code) & (pl.col('stand') == 'L')
             ).count() /
             pl.col('pitch_type').filter(pl.col('stand') == 'L').count().clip(lower_bound=1)
            ).alias(f'{abbr}_usage_vL')
            for code, abbr in PITCH_TYPES_FULL.items()
        ],
    ])
    .sort(['game_date'])
)

print(f"Game-level pitcher agg shape: {df_game_aggs.shape}")

Game-level pitcher agg shape: (14352, 84)


In [30]:
df_game_aggs

game_date,player_name,away_team,home_team,p_throws,is_opener_game,season,abs_era,Outs,Pitches,PA,Runs,Hits,K,BB,HBP,HR,FB,GB,Strikes,Balls,BIP,Whiffs,CS,CSW,xBA,wOBA,xwOBA,throws_ff,throws_si,throws_fc,throws_sl,throws_st,throws_cu,throws_ch,ff_velo,ff_spinrate,…,fc_ivb,fc_hb,fc_vaa,sl_velo,sl_spinrate,sl_ivb,sl_hb,sl_vaa,st_velo,st_spinrate,st_ivb,st_hb,st_vaa,cu_velo,cu_spinrate,cu_ivb,cu_hb,cu_vaa,ch_velo,ch_spinrate,ch_ivb,ch_hb,ch_vaa,ff_usage_vR,si_usage_vR,fc_usage_vR,sl_usage_vR,st_usage_vR,cu_usage_vR,ch_usage_vR,ff_usage_vL,si_usage_vL,fc_usage_vL,sl_usage_vL,st_usage_vL,cu_usage_vL,ch_usage_vL
datetime[μs],str,str,str,str,i8,i32,i8,u32,u32,u32,i64,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,i8,i8,i8,i8,i8,i8,i8,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2023-03-30 00:00:00,"""Aaron Nola""","""PHI""","""TEX""","""R""",0,2023,0,11,72,17,5,4,4,2,0,1,2,4,37,24,11,5,14,19,0.399364,0.379412,0.358283,1,1,1,0,0,0,1,92.365625,2367.90625,…,11760.274286,2991.908571,-7.212637,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,85.422222,1577.777778,15528.96,-25989.12,-7.031837,0.461538,0.076923,0.269231,0.0,0.0,0.0,0.038462,0.434783,0.043478,0.0,0.0,0.0,0.0,0.173913
2023-03-30 00:00:00,"""Alek Manoah""","""TOR""","""STL""","""R""",0,2023,0,10,86,21,2,9,3,2,0,2,4,7,38,32,16,12,14,26,0.4338125,0.557143,0.438848,1,1,0,1,0,0,1,94.165714,2394.657143,…,null,null,null,81.565,2258.9,404.352,25691.904,-9.018129,null,null,null,null,null,null,null,null,null,null,87.028571,2082.142857,10723.474286,-27638.125714,-8.060548,0.390244,0.317073,0.0,0.292683,0.0,0.0,0.0,0.431818,0.227273,0.0,0.181818,0.0,0.0,0.159091
2023-03-30 00:00:00,"""Blake Snell""","""COL""","""SD""","""L""",0,2023,0,13,93,21,2,6,9,1,0,0,1,4,45,37,11,19,10,29,0.395818,0.366667,0.235329,1,0,0,1,0,1,1,94.929231,2391.4,…,null,null,null,89.225,2324.125,19854.72,-3680.64,-9.365208,null,null,null,null,null,81.745455,2378.090909,-17531.345455,-23921.803636,-12.87579,87.233333,1769.444444,20298.24,23339.52,-8.203635,0.634921,0.0,0.0,0.063492,0.0,0.15873,0.142857,0.833333,0.0,0.0,0.133333,0.0,0.033333,0.0
2023-03-30 00:00:00,"""Corbin Burnes""","""MIL""","""CHC""","""R""",0,2023,0,15,87,23,0,4,3,3,1,0,3,11,39,32,16,7,14,21,0.23625,0.317391,0.302161,0,1,1,1,0,1,1,null,null,…,21118.464,5746.176,-6.314724,87.335714,2824.285714,-562.834286,11923.2,-8.973709,null,null,null,null,null,80.61,2776.7,-12960.0,19035.648,-10.594777,88.283333,2110.333333,10609.92,-25159.68,-7.4953,0.0,0.146341,0.390244,0.341463,0.0,0.097561,0.02439,0.0,0.0,0.630435,0.0,0.0,0.130435,0.23913
2023-03-30 00:00:00,"""Corey Kluber""","""BAL""","""BOS""","""R""",0,2023,0,9,80,19,1,6,4,4,0,2,3,2,37,32,11,6,11,17,0.526091,0.584211,0.515401,1,1,1,0,0,1,1,87.666667,2314.666667,…,17553.024,4561.92,-5.832406,null,null,null,null,null,null,null,null,null,null,79.790476,2544.5,9410.194286,21802.422857,-7.908717,81.966667,1724.5,10460.16,-29952.0,-8.402486,0.0,0.25,0.321429,0.0,0.0,0.392857,0.035714,0.057692,0.384615,0.211538,0.0,0.0,0.192308,0.153846
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-09-29 00:00:00,"""Xzavion Curry""","""MIA""","""TOR""","""R""",0,2025,0,15,65,17,0,2,1,1,0,1,5,5,27,23,15,2,15,17,0.2352,0.211765,0.258184,1,0,0,1,0,1,1,92.216667,2039.638889,…,null,null,null,83.3375,2335.4375,-920.16,13335.84,-8.703572,null,null,null,null,null,73.55,2461.416667,-22913.28,20338.56,-12.229459,84.3,1071.0,20113.92,-18455.04,-4.978235,0.487179,0.0,0.0,0.384615,0.0,0.128205,0.0,0.653846,0.0,0.0,0.038462,0.0,0.269231,0.038462
2025-09-29 00:00:00,"""Yariel Rodriguez""","""MIA""","""TOR""","""R""",0,2025,0,15,82,21,1,4,5,2,0,0,3,6,40,28,14,9,14,23,0.210571,0.271429,0.208203,1,1,0,1,0,1,1,93.684848,2287.121212,…,null,null,null,84.42381,2627.714286,898.56,16717.16571

In [73]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
df_game_aggs.write_parquet(PROCESSED_DATA_DIR / "Pitcher2023-2025.parquet")

In [ ]:
df_game_aggs.write_csv(PROCESSED_DATA_DIR / "Pitcher2023-2025.csv")

: 